# Experiment 7 — Autoencoders, Convolutional AE, Denoising AE & VAE
CS3807 — Deep Learning Laboratory

Runs end-to-end in Google Colab (**Runtime → Change runtime type → GPU** recommended,
though everything here is small enough to also finish on CPU).

Produces every result the lab report needs: FC autoencoder, Convolutional
Autoencoder, Denoising CAE (Gaussian + salt-and-pepper noise, multi-level
study), a 2-D Variational Autoencoder (latent space, generation, interpolation),
the latent-dimension sweep, reconstruction-error analysis, and all consolidated
tables. Everything is saved to `results/summary.json` + `results/figures/`
and zipped for download at the end.

In [ ]:
!pip -q install scikit-image seaborn > /dev/null
print("Dependencies installed.")

In [ ]:
import os, json, time, shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.metrics import structural_similarity as ssim_fn

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks, backend as K

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

RESULTS_DIR = Path("results"); FIG_DIR = RESULTS_DIR / "figures"
for d in [RESULTS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUMMARY = {}
print("GPU available:", tf.config.list_physical_devices('GPU'))

# ---- Load MNIST (built into Keras -- no external download needed) ----
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.mnist.load_data()

rng = np.random.RandomState(SEED)
USE_FULL_DATASET = True  # set to False to fall back to the 10k/2k recommended subset

if USE_FULL_DATASET:
    x_train = x_train_full.astype("float32") / 255.0
    y_train = y_train_full
    x_test = x_test_full.astype("float32") / 255.0
    y_test = y_test_full
else:
    N_TRAIN, N_TEST = 10000, 2000   # recommended laboratory subset
    train_idx = rng.choice(len(x_train_full), N_TRAIN, replace=False)
    test_idx = rng.choice(len(x_test_full), N_TEST, replace=False)
    x_train = x_train_full[train_idx].astype("float32") / 255.0
    y_train = y_train_full[train_idx]
    x_test = x_test_full[test_idx].astype("float32") / 255.0
    y_test = y_test_full[test_idx]

N_TRAIN, N_TEST = len(x_train), len(x_test)

# carve a validation split out of the training set
n_val = int(0.1 * N_TRAIN)
perm = rng.permutation(N_TRAIN)
val_idx2, train_idx2 = perm[:n_val], perm[n_val:]
x_val = x_train[val_idx2]
x_train_final = x_train[train_idx2]

x_train_img = x_train_final[..., np.newaxis]          # (N, 28, 28, 1)
x_val_img = x_val[..., np.newaxis]
x_test_img = x_test[..., np.newaxis]

x_train_flat = x_train_final.reshape(len(x_train_final), 784)
x_val_flat = x_val.reshape(len(x_val), 784)
x_test_flat = x_test.reshape(len(x_test), 784)

SUMMARY["dataset"] = {
    "name": "MNIST Handwritten Digit Dataset",
    "n_train": int(len(x_train_final)), "n_val": int(len(x_val)), "n_test": int(len(x_test)),
    "image_shape": [28, 28, 1], "pixel_range_used": "[0, 1] (divided by 255)",
    "flattened_dim": 784,
}
print(json.dumps(SUMMARY["dataset"], indent=2))

In [ ]:
def compute_metrics(x_true, x_pred):
    """x_true, x_pred: (N, 28, 28, 1) or (N, 784) arrays in [0,1]. Returns MSE, MAE, mean SSIM."""
    xt = x_true.reshape(len(x_true), 28, 28)
    xp = x_pred.reshape(len(x_pred), 28, 28)
    mse = float(np.mean((xt - xp) ** 2))
    mae = float(np.mean(np.abs(xt - xp)))
    ssims = [ssim_fn(xt[i], xp[i], data_range=1.0) for i in range(len(xt))]
    return mse, mae, float(np.mean(ssims))

def per_image_mse(x_true, x_pred):
    xt = x_true.reshape(len(x_true), -1)
    xp = x_pred.reshape(len(x_pred), -1)
    return np.mean((xt - xp) ** 2, axis=1)

def save_recon_grid(originals, reconstructions, path, n=10, titles=("Original", "Reconstructed")):
    fig, axes = plt.subplots(2, n, figsize=(n * 1.1, 2.4))
    for i in range(n):
        axes[0, i].imshow(originals[i].reshape(28, 28), cmap="gray"); axes[0, i].axis("off")
        axes[1, i].imshow(reconstructions[i].reshape(28, 28), cmap="gray"); axes[1, i].axis("off")
    axes[0, 0].set_ylabel(titles[0], fontsize=9); axes[1, 0].set_ylabel(titles[1], fontsize=9)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

def save_triplet_grid(a, b, c, path, n=10, row_labels=("Clean", "Noisy", "Denoised")):
    fig, axes = plt.subplots(3, n, figsize=(n * 1.1, 3.5))
    for i in range(n):
        axes[0, i].imshow(a[i].reshape(28, 28), cmap="gray"); axes[0, i].axis("off")
        axes[1, i].imshow(b[i].reshape(28, 28), cmap="gray"); axes[1, i].axis("off")
        axes[2, i].imshow(c[i].reshape(28, 28), cmap="gray"); axes[2, i].axis("off")
    for r, lbl in enumerate(row_labels):
        axes[r, 0].set_title(lbl, fontsize=9, loc="left")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

## 1. Fully Connected Autoencoder

In [ ]:
def build_fc_autoencoder(latent_dim=16):
    inp = layers.Input(shape=(784,))
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dense(32, activation="relu")(x)
    z = layers.Dense(latent_dim, activation="relu", name="latent")(x)
    x = layers.Dense(32, activation="relu")(z)
    x = layers.Dense(128, activation="relu")(x)
    out = layers.Dense(784, activation="sigmoid")(x)
    return models.Model(inp, out, name=f"FC_Autoencoder_z{latent_dim}")

fc_model = build_fc_autoencoder(latent_dim=16)
fc_model.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy")
fc_params = fc_model.count_params()

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
t0 = time.time()
fc_history = fc_model.fit(x_train_flat, x_train_flat, validation_data=(x_val_flat, x_val_flat),
                           epochs=20, batch_size=128, callbacks=[es], verbose=2)
fc_train_time = time.time() - t0

fc_recon_flat = fc_model.predict(x_test_flat, verbose=0)
fc_mse, fc_mae, fc_ssim = compute_metrics(x_test_flat, fc_recon_flat)
print(f"FC-AE: MSE={fc_mse:.5f} MAE={fc_mae:.5f} SSIM={fc_ssim:.4f} params={fc_params} time={fc_train_time:.1f}s")

# Plot 1: Original vs Reconstructed
plot1_path = FIG_DIR / "fc_ae_original_vs_reconstructed.png"
save_recon_grid(x_test_flat, fc_recon_flat, plot1_path)

# Plot 2: Training/validation loss
plot2_path = FIG_DIR / "fc_ae_loss.png"
plt.figure(figsize=(6, 4))
plt.plot(fc_history.history["loss"], label="train_loss")
plt.plot(fc_history.history["val_loss"], label="val_loss")
plt.title("FC Autoencoder Training/Validation Loss"); plt.xlabel("epoch"); plt.ylabel("BCE loss"); plt.legend()
plt.tight_layout(); plt.savefig(plot2_path, dpi=150); plt.show()

fc_result = {
    "architecture": "784-128-32-16-32-128-784, ReLU hidden, sigmoid output",
    "latent_dimension": 16, "optimizer": "Adam", "learning_rate": 1e-3,
    "batch_size": 128, "epochs_configured": 20, "epochs_run": len(fc_history.history["loss"]),
    "loss_function": "Binary Cross-Entropy", "parameters": int(fc_params),
    "training_time_sec": round(fc_train_time, 2),
    "test_mse": round(fc_mse, 6), "test_mae": round(fc_mae, 6), "mean_ssim": round(fc_ssim, 4),
    "figures": {"original_vs_reconstructed": str(plot1_path), "loss_curve": str(plot2_path)},
}
SUMMARY["fc_autoencoder"] = fc_result

## 2. Convolutional Autoencoder

In [ ]:
def build_conv_autoencoder():
    inp = layers.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(inp)
    x = layers.MaxPooling2D(2, padding="same")(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2, padding="same")(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same", name="latent_feature_map")(x)
    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.UpSampling2D(2)(x)
    out = layers.Conv2D(1, 3, activation="sigmoid", padding="same")(x)
    return models.Model(inp, out, name="Convolutional_Autoencoder")

conv_model = build_conv_autoencoder()
conv_model.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy")
conv_params = conv_model.count_params()

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
t0 = time.time()
conv_history = conv_model.fit(x_train_img, x_train_img, validation_data=(x_val_img, x_val_img),
                               epochs=20, batch_size=128, callbacks=[es], verbose=2)
conv_train_time = time.time() - t0

conv_recon_img = conv_model.predict(x_test_img, verbose=0)
conv_mse, conv_mae, conv_ssim = compute_metrics(x_test_img, conv_recon_img)
print(f"Conv-AE: MSE={conv_mse:.5f} MAE={conv_mae:.5f} SSIM={conv_ssim:.4f} params={conv_params} time={conv_train_time:.1f}s")

# Plot 3: FC-AE vs CAE reconstruction comparison
plot3_path = FIG_DIR / "fc_vs_conv_ae_comparison.png"
n = 8
fig, axes = plt.subplots(3, n, figsize=(n * 1.1, 3.5))
for i in range(n):
    axes[0, i].imshow(x_test_img[i].reshape(28, 28), cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(fc_recon_flat[i].reshape(28, 28), cmap="gray"); axes[1, i].axis("off")
    axes[2, i].imshow(conv_recon_img[i].reshape(28, 28), cmap="gray"); axes[2, i].axis("off")
axes[0, 0].set_title("Original", fontsize=9, loc="left")
axes[1, 0].set_title("FC-AE", fontsize=9, loc="left")
axes[2, 0].set_title("Conv-AE", fontsize=9, loc="left")
plt.tight_layout(); plt.savefig(plot3_path, dpi=150); plt.show()

conv_result = {
    "architecture": "Conv2D(32)-MaxPool-Conv2D(64)-MaxPool-Conv2D(64)-UpSampling-Conv2D(32)-UpSampling-Conv2D(1,sigmoid)",
    "optimizer": "Adam", "learning_rate": 1e-3, "batch_size": 128,
    "epochs_configured": 20, "epochs_run": len(conv_history.history["loss"]),
    "loss_function": "Binary Cross-Entropy", "parameters": int(conv_params),
    "training_time_sec": round(conv_train_time, 2),
    "test_mse": round(conv_mse, 6), "test_mae": round(conv_mae, 6), "mean_ssim": round(conv_ssim, 4),
    "figures": {"fc_vs_conv_comparison": str(plot3_path)},
}
SUMMARY["conv_autoencoder"] = conv_result
SUMMARY["fc_vs_conv_comparison_table"] = {
    "Fully Connected AE": {"MSE": fc_result["test_mse"], "MAE": fc_result["test_mae"],
                            "SSIM": fc_result["mean_ssim"], "Parameters": fc_result["parameters"]},
    "Convolutional AE": {"MSE": conv_result["test_mse"], "MAE": conv_result["test_mae"],
                          "SSIM": conv_result["mean_ssim"], "Parameters": conv_result["parameters"]},
}

## 3. Denoising Autoencoder (Gaussian + Salt-and-Pepper Noise)

In [ ]:
def add_gaussian_noise(x, sigma, rng_local=None):
    rng_local = rng_local or np.random.RandomState(SEED)
    noisy = x + rng_local.normal(0.0, sigma, size=x.shape).astype("float32")
    return np.clip(noisy, 0.0, 1.0)

def add_salt_pepper_noise(x, p, rng_local=None):
    rng_local = rng_local or np.random.RandomState(SEED)
    noisy = x.copy()
    mask = rng_local.rand(*x.shape)
    noisy[mask < (p / 2)] = 0.0
    noisy[(mask >= (p / 2)) & (mask < p)] = 1.0
    return noisy

def train_denoising_cae(sigma, epochs=15):
    rng_local = np.random.RandomState(SEED + int(sigma * 100))
    x_train_noisy = add_gaussian_noise(x_train_img, sigma, rng_local)
    x_val_noisy = add_gaussian_noise(x_val_img, sigma, rng_local)
    model = build_conv_autoencoder()
    model._name = f"Denoising_CAE_sigma{sigma}"
    model.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy")
    es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    t0 = time.time()
    hist = model.fit(x_train_noisy, x_train_img, validation_data=(x_val_noisy, x_val_img),
                      epochs=epochs, batch_size=128, callbacks=[es], verbose=2)
    train_time = time.time() - t0
    x_test_noisy = add_gaussian_noise(x_test_img, sigma, rng_local)
    recon = model.predict(x_test_noisy, verbose=0)
    mse, mae, ssim_v = compute_metrics(x_test_img, recon)
    return model, hist, train_time, x_test_noisy, recon, mse, mae, ssim_v

GAUSSIAN_LEVELS = [0.1, 0.2, 0.3]
denoising_models, noise_level_table = {}, {}
for sigma in GAUSSIAN_LEVELS:
    print(f"--- Training denoising CAE for Gaussian sigma={sigma} ---")
    model, hist, ttime, x_noisy, recon, mse, mae, ssim_v = train_denoising_cae(sigma)
    denoising_models[sigma] = {"model": model, "history": hist, "train_time": ttime,
                                "x_noisy": x_noisy, "recon": recon}
    noise_level_table[str(sigma)] = {"MSE": round(mse, 6), "MAE": round(mae, 6), "SSIM": round(ssim_v, 4)}
    print(f"  sigma={sigma}: MSE={mse:.5f} MAE={mae:.5f} SSIM={ssim_v:.4f} time={ttime:.1f}s")

SUMMARY["denoising_gaussian_noise_levels"] = noise_level_table

# Plot 4: Clean vs Noisy vs Denoised (primary sigma = 0.2)
primary_sigma = 0.2
plot4_path = FIG_DIR / "denoising_clean_noisy_denoised.png"
save_triplet_grid(x_test_img, denoising_models[primary_sigma]["x_noisy"],
                   denoising_models[primary_sigma]["recon"], plot4_path)

# Plot 5: Noise level vs MSE/MAE/SSIM
plot5_path = FIG_DIR / "denoising_noise_level_vs_quality.png"
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
metrics_names = ["MSE", "MAE", "SSIM"]
for ax, m in zip(axes, metrics_names):
    vals = [noise_level_table[str(s)][m] for s in GAUSSIAN_LEVELS]
    ax.plot(GAUSSIAN_LEVELS, vals, marker="o")
    ax.set_xlabel("Gaussian noise sigma"); ax.set_ylabel(m); ax.set_title(f"Noise level vs {m}")
plt.tight_layout(); plt.savefig(plot5_path, dpi=150); plt.show()

primary_result = {
    "primary_training_noise": primary_sigma,
    "note": "Each Gaussian noise level was trained as its own denoising CAE "
            "(same Conv-AE architecture); the sigma=0.2 model is used for the "
            "clean/noisy/denoised qualitative figure.",
    "architecture": conv_result["architecture"],
    "parameters": conv_result["parameters"],
    "training_time_sec_per_level": {str(s): round(denoising_models[s]["train_time"], 2) for s in GAUSSIAN_LEVELS},
    "figures": {"clean_noisy_denoised": str(plot4_path), "noise_level_vs_quality": str(plot5_path)},
}
SUMMARY["denoising_cae"] = primary_result

In [ ]:
# Salt-and-pepper noise -- second corruption type (qualitative demonstration)
SP_LEVEL = 0.10
rng_sp = np.random.RandomState(SEED + 999)
x_train_sp = add_salt_pepper_noise(x_train_img, SP_LEVEL, rng_sp)
x_val_sp = add_salt_pepper_noise(x_val_img, SP_LEVEL, rng_sp)
x_test_sp = add_salt_pepper_noise(x_test_img, SP_LEVEL, rng_sp)

sp_model = build_conv_autoencoder(); sp_model._name = "Denoising_CAE_saltpepper"
sp_model.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy")
es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
t0 = time.time()
sp_hist = sp_model.fit(x_train_sp, x_train_img, validation_data=(x_val_sp, x_val_img),
                        epochs=15, batch_size=128, callbacks=[es], verbose=2)
sp_time = time.time() - t0
sp_recon = sp_model.predict(x_test_sp, verbose=0)
sp_mse, sp_mae, sp_ssim = compute_metrics(x_test_img, sp_recon)
print(f"Salt-and-pepper (p={SP_LEVEL}): MSE={sp_mse:.5f} MAE={sp_mae:.5f} SSIM={sp_ssim:.4f} time={sp_time:.1f}s")

plot4b_path = FIG_DIR / "denoising_saltpepper_clean_noisy_denoised.png"
save_triplet_grid(x_test_img, x_test_sp, sp_recon, plot4b_path)

SUMMARY["denoising_salt_and_pepper"] = {
    "corruption_level_p": SP_LEVEL, "parameters": int(sp_model.count_params()),
    "training_time_sec": round(sp_time, 2),
    "test_mse": round(sp_mse, 6), "test_mae": round(sp_mae, 6), "mean_ssim": round(sp_ssim, 4),
    "figure": str(plot4b_path),
}

## 4. Variational Autoencoder (latent dimension = 2)

In [ ]:
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

LATENT_DIM_VAE = 2

def build_vae_encoder(latent_dim=LATENT_DIM_VAE):
    inp = layers.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, 3, strides=2, activation="relu", padding="same")(inp)
    x = layers.Conv2D(64, 3, strides=2, activation="relu", padding="same")(x)
    x = layers.Flatten()(x)
    x = layers.Dense(16, activation="relu")(x)
    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling()([z_mean, z_log_var])
    return models.Model(inp, [z_mean, z_log_var, z], name="VAE_Encoder")

def build_vae_decoder(latent_dim=LATENT_DIM_VAE):
    inp = layers.Input(shape=(latent_dim,))
    x = layers.Dense(7 * 7 * 64, activation="relu")(inp)
    x = layers.Reshape((7, 7, 64))(x)
    x = layers.Conv2DTranspose(64, 3, strides=2, activation="relu", padding="same")(x)
    x = layers.Conv2DTranspose(32, 3, strides=2, activation="relu", padding="same")(x)
    out = layers.Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(x)
    return models.Model(inp, out, name="VAE_Decoder")

class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder, self.decoder = encoder, decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.rec_loss_tracker = tf.keras.metrics.Mean(name="rec_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.rec_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x)
            recon = self.decoder(z)
            rec_loss = tf.reduce_mean(tf.reduce_sum(
                tf.keras.losses.binary_crossentropy(x, recon), axis=(1, 2)))
            kl_loss = -0.5 * tf.reduce_mean(tf.reduce_sum(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1))
            total_loss = rec_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.rec_loss_tracker.update_state(rec_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        z_mean, z_log_var, z = self.encoder(x)
        recon = self.decoder(z)
        rec_loss = tf.reduce_mean(tf.reduce_sum(
            tf.keras.losses.binary_crossentropy(x, recon), axis=(1, 2)))
        kl_loss = -0.5 * tf.reduce_mean(tf.reduce_sum(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1))
        total_loss = rec_loss + kl_loss
        self.total_loss_tracker.update_state(total_loss)
        self.rec_loss_tracker.update_state(rec_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

vae_encoder = build_vae_encoder()
vae_decoder = build_vae_decoder()
vae = VAE(vae_encoder, vae_decoder)
vae.compile(optimizer=optimizers.Adam(1e-3))
vae_params = vae_encoder.count_params() + vae_decoder.count_params()

val_ds_vae = tf.data.Dataset.from_tensor_slices(x_val_img).batch(128)
t0 = time.time()
vae_history = vae.fit(x_train_img, validation_data=val_ds_vae, epochs=30, batch_size=128, verbose=2)
vae_train_time = time.time() - t0

# Plot 9: VAE training/validation reconstruction (total) loss
plot9_path = FIG_DIR / "vae_training_validation_loss.png"
plt.figure(figsize=(6, 4))
plt.plot(vae_history.history["loss"], label="train_total_loss")
plt.plot(vae_history.history["val_loss"], label="val_total_loss")
plt.title("VAE Training/Validation Loss"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.tight_layout(); plt.savefig(plot9_path, dpi=150); plt.show()

# Test-set evaluation using the mean mu (deterministic reconstruction for metrics)
z_mean_test, z_log_var_test, _ = vae_encoder.predict(x_test_img, verbose=0)
vae_recon_test = vae_decoder.predict(z_mean_test, verbose=0)
vae_mse, vae_mae, vae_ssim = compute_metrics(x_test_img, vae_recon_test)

final_rec_loss = float(vae_history.history["rec_loss"][-1])
final_kl_loss = float(vae_history.history["kl_loss"][-1])
final_total_loss = float(vae_history.history["loss"][-1])

print(f"VAE: recon_loss={final_rec_loss:.3f} KL_loss={final_kl_loss:.3f} total={final_total_loss:.3f}")
print(f"VAE test-set: MSE={vae_mse:.5f} MAE={vae_mae:.5f} SSIM={vae_ssim:.4f} params={vae_params} time={vae_train_time:.1f}s")

vae_result = {
    "latent_dimension": LATENT_DIM_VAE,
    "encoder": "Conv2D(32,s2)-Conv2D(64,s2)-Flatten-Dense(16)-[mu,logvar]-Sampling",
    "decoder": "Dense(7*7*64)-Reshape-Conv2DTranspose(64,s2)-Conv2DTranspose(32,s2)-Conv2DTranspose(1,sigmoid)",
    "loss": "Binary cross-entropy reconstruction + analytic KL divergence to N(0,I)",
    "optimizer": "Adam", "learning_rate": 1e-3, "batch_size": 128,
    "epochs_configured": 30, "epochs_run": len(vae_history.history["loss"]),
    "parameters": int(vae_params), "training_time_sec": round(vae_train_time, 2),
    "final_reconstruction_loss": round(final_rec_loss, 4),
    "final_kl_loss": round(final_kl_loss, 4),
    "final_total_loss": round(final_total_loss, 4),
    "test_mse": round(vae_mse, 6), "test_mae": round(vae_mae, 6), "mean_ssim": round(vae_ssim, 4),
    "figures": {"training_validation_loss": str(plot9_path)},
}
SUMMARY["vae"] = vae_result

In [ ]:
# Plot 6: VAE latent space (2D), colored by digit label
plot6_path = FIG_DIR / "vae_latent_space.png"
plt.figure(figsize=(7, 6))
scatter = plt.scatter(z_mean_test[:, 0], z_mean_test[:, 1], c=y_test, cmap="tab10", s=8, alpha=0.7)
plt.colorbar(scatter, label="digit label")
plt.xlabel("z1"); plt.ylabel("z2"); plt.title("VAE Latent Space (test set, colored by digit label)")
plt.tight_layout(); plt.savefig(plot6_path, dpi=150); plt.show()

# Plot 7: Randomly generated images (25) by sampling z ~ N(0, I)
plot7_path = FIG_DIR / "vae_random_generated.png"
rng_gen = np.random.RandomState(SEED)
z_random = rng_gen.normal(0, 1, size=(25, LATENT_DIM_VAE)).astype("float32")
generated = vae_decoder.predict(z_random, verbose=0)
fig, axes = plt.subplots(5, 5, figsize=(6, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].reshape(28, 28), cmap="gray"); ax.axis("off")
plt.suptitle("VAE Randomly Generated Images (z ~ N(0, I))")
plt.tight_layout(); plt.savefig(plot7_path, dpi=150); plt.show()

# Plot 8: Latent-space interpolation between two real test-image encodings
plot8_path = FIG_DIR / "vae_latent_interpolation.png"
zA, zB = z_mean_test[0], z_mean_test[1]
alphas = np.linspace(0, 1, 11)
z_interp = np.stack([(1 - a) * zA + a * zB for a in alphas])
decoded_interp = vae_decoder.predict(z_interp, verbose=0)
fig, axes = plt.subplots(1, len(alphas), figsize=(len(alphas) * 1.1, 1.5))
for i, ax in enumerate(axes):
    ax.imshow(decoded_interp[i].reshape(28, 28), cmap="gray"); ax.axis("off")
    ax.set_title(f"{alphas[i]:.1f}", fontsize=7)
plt.suptitle(f"Latent Interpolation: digit {y_test[0]} -> digit {y_test[1]}")
plt.tight_layout(); plt.savefig(plot8_path, dpi=150); plt.show()

SUMMARY["vae"]["figures"].update({
    "latent_space": str(plot6_path),
    "random_generated": str(plot7_path),
    "latent_interpolation": str(plot8_path),
    "interpolation_endpoints_digits": [int(y_test[0]), int(y_test[1])],
})

## 5. Reconstruction Error Distribution & High-Error Samples (VAE)

In [ ]:
errors = per_image_mse(x_test_img, vae_recon_test)

plot_err_path = FIG_DIR / "reconstruction_error_distribution.png"
plt.figure(figsize=(6, 4))
plt.hist(errors, bins=40)
plt.xlabel("Per-image reconstruction MSE"); plt.ylabel("Count")
plt.title("Reconstruction Error Distribution (VAE, test set)")
plt.tight_layout(); plt.savefig(plot_err_path, dpi=150); plt.show()

top5_idx = np.argsort(errors)[-5:][::-1]
plot_high_err_path = FIG_DIR / "high_error_reconstructions.png"
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for col, idx in enumerate(top5_idx):
    axes[0, col].imshow(x_test_img[idx].reshape(28, 28), cmap="gray"); axes[0, col].axis("off")
    axes[1, col].imshow(vae_recon_test[idx].reshape(28, 28), cmap="gray"); axes[1, col].axis("off")
    axes[0, col].set_title(f"err={errors[idx]:.4f}\ndigit={y_test[idx]}", fontsize=8)
axes[0, 0].set_ylabel("Original", fontsize=9); axes[1, 0].set_ylabel("Reconstruction", fontsize=9)
plt.suptitle("Top-5 Highest-Error Test Images (VAE)")
plt.tight_layout(); plt.savefig(plot_high_err_path, dpi=150); plt.show()

SUMMARY["reconstruction_error_analysis"] = {
    "model_used": "VAE",
    "mean_error": round(float(np.mean(errors)), 6),
    "std_error": round(float(np.std(errors)), 6),
    "min_error": round(float(np.min(errors)), 6),
    "max_error": round(float(np.max(errors)), 6),
    "top5_high_error_samples": [
        {"index": int(idx), "digit_label": int(y_test[idx]), "mse": round(float(errors[idx]), 6)}
        for idx in top5_idx
    ],
    "figures": {"error_distribution": str(plot_err_path), "high_error_examples": str(plot_high_err_path)},
}
print(SUMMARY["reconstruction_error_analysis"])

## 6. Additional Latent-Dimension Study (FC Autoencoder, d_z ∈ {2, 8, 16, 32})

In [ ]:
latent_dims = [2, 8, 16, 32]
latent_dim_table = {}
for dz in latent_dims:
    print(f"--- Training FC-AE with latent dimension {dz} ---")
    m = build_fc_autoencoder(latent_dim=dz)
    m.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy")
    es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    m.fit(x_train_flat, x_train_flat, validation_data=(x_val_flat, x_val_flat),
          epochs=15, batch_size=128, callbacks=[es], verbose=0)
    recon = m.predict(x_test_flat, verbose=0)
    mse, mae, ssim_v = compute_metrics(x_test_flat, recon)
    latent_dim_table[str(dz)] = {"MSE": round(mse, 6), "SSIM": round(ssim_v, 4), "Parameters": int(m.count_params())}
    print(f"  d_z={dz}: MSE={mse:.5f} SSIM={ssim_v:.4f} params={m.count_params()}")

plot10_path = FIG_DIR / "latent_dimension_vs_mse.png"
plt.figure(figsize=(6, 4))
plt.plot(latent_dims, [latent_dim_table[str(d)]["MSE"] for d in latent_dims], marker="o")
plt.xlabel("Latent dimension"); plt.ylabel("Test MSE")
plt.title("Latent Dimension vs. Reconstruction MSE")
plt.tight_layout(); plt.savefig(plot10_path, dpi=150); plt.show()

SUMMARY["latent_dimension_study"] = {"results": latent_dim_table, "figure": str(plot10_path)}

## 7. Consolidated Results

In [ ]:
consolidated = {
    "FC Autoencoder": {"MSE": fc_result["test_mse"], "MAE": fc_result["test_mae"],
                        "SSIM": fc_result["mean_ssim"], "Parameters": fc_result["parameters"],
                        "Training_Time_sec": fc_result["training_time_sec"]},
    "Convolutional AE": {"MSE": conv_result["test_mse"], "MAE": conv_result["test_mae"],
                          "SSIM": conv_result["mean_ssim"], "Parameters": conv_result["parameters"],
                          "Training_Time_sec": conv_result["training_time_sec"]},
    "Denoising CAE (sigma=0.2)": {
        "MSE": noise_level_table["0.2"]["MSE"], "MAE": noise_level_table["0.2"]["MAE"],
        "SSIM": noise_level_table["0.2"]["SSIM"], "Parameters": primary_result["parameters"],
        "Training_Time_sec": primary_result["training_time_sec_per_level"]["0.2"],
    },
    "VAE": {"MSE": vae_result["test_mse"], "MAE": vae_result["test_mae"],
            "SSIM": vae_result["mean_ssim"], "Parameters": vae_result["parameters"],
            "Training_Time_sec": vae_result["training_time_sec"]},
}
SUMMARY["consolidated_results"] = consolidated
print(json.dumps(consolidated, indent=2))

with open(RESULTS_DIR / "summary.json", "w") as f:
    json.dump(SUMMARY, f, indent=2)
print("Saved:", (RESULTS_DIR / "summary.json").resolve())

## Download everything

In [ ]:
shutil.make_archive("experiment7_results", "zip", RESULTS_DIR)
print("Created experiment7_results.zip")

try:
    from google.colab import files
    files.download("experiment7_results.zip")
except ImportError:
    print("Not running in Colab — find experiment7_results.zip in the working directory.")